# Experiment 18 — LoREx-160 controlled prompt and JSON-recovery ablations

**Design.** Same 135 saved clean Tesseract OCR records, Qwen/Qwen2.5-7B-Instruct, Featherless AI, `temperature=0`, `max_tokens=300`, and a shared recovery parser. Five inference variants: **A0** full P2; **A1** P2 minus load-number guidance; **A2** P2 minus field descriptions; **A3** P2 minus output-format/missing-value instructions; **A4** original P1 at temperature 0. **A5** is *not* a sixth inference run: evaluate A0 raw responses with strict JSON parsing instead of recovery parsing.

The existing P1/P2 predictions are historical results, **not** controlled ablations (historical P1 omitted temperature). Raw responses, failures and prompts are retained. A failed API request is not counted as a successful empty extraction. Default pilot size is 10; change `LIMIT=135` for the complete matched set. Results are written under `experiments/prompt_ablation/` and never overwrite historical predictions.

**Run this notebook from the repository root** (`~/lorex-ocr-llm-experiments`) with its `.venv` kernel. This notebook requires the local repository scripts and saved OCR file; the uploaded dataset splits alone are insufficient for inference.

In [ ]:
from pathlib import Path
import ast, csv, difflib, json, os, re, time
from collections import Counter
from datetime import datetime, timezone
from statistics import mean

from dotenv import load_dotenv
from huggingface_hub import InferenceClient

ROOT = Path.cwd()
assert (ROOT / 'scripts/exp_03_lorex_tesseract_llm_prompt2.py').exists(), 'Set notebook working directory to repository root'
OCR_FILE = ROOT / 'experiments/matched_inference/ocr/clean/tesseract_clean_135.json'
OUT_DIR = ROOT / 'experiments/prompt_ablation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LIMIT = 10  # pilot; set to 135 for full matched subset
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
PROVIDER = 'featherless-ai'
MAX_TOKENS = 300
TEMPERATURE = 0
FIELDS = ['load_number','pickup_location','pickup_time','dropoff_location','dropoff_time','total_rate','rate_per_mile']
VARIANTS = ['A0_full_P2','A1_no_load_guidance','A2_no_field_guidance','A3_no_format_null_rules','A4_P1_controlled']
load_dotenv()
ocr_rows = json.loads(OCR_FILE.read_text(encoding='utf-8'))
assert len(ocr_rows) == 135
keys = [(r['split'],r['source_image']) for r in ocr_rows]
assert len(set(keys)) == 135
print('OCR records:',len(ocr_rows),'pilot:',LIMIT,'output:',OUT_DIR)

## Prompt construction and audit
The two original prompt functions are extracted from their source scripts without importing OCR/API initialization. Ablations delete only the specified instruction lines. A3 **retains the JSON key schema**, because removing the schema would change the task definition rather than isolate formatting instructions. Inspect the printed prompt diffs before inference.

In [ ]:
def load_prompt_function(script_name):
    path = ROOT / 'scripts' / script_name
    tree = ast.parse(path.read_text(encoding='utf-8'))
    funcs = [x for x in tree.body if isinstance(x, ast.FunctionDef) and x.name == 'build_prompt']
    assert len(funcs) == 1
    module = ast.Module(body=funcs, type_ignores=[])
    ns = {}
    exec(compile(module, str(path), 'exec'), ns)
    return ns['build_prompt']

p1 = load_prompt_function('exp_03_lorex_tesseract_llm_prompt1.py')
p2 = load_prompt_function('exp_03_lorex_tesseract_llm_prompt2.py')

LOAD_LINES = [
    '- The load_number may NOT be explicitly labeled as "load number".',
    '- It may appear as a short standalone alphanumeric code near the top of the document or screenshot.',
    '- It may look like examples such as: 116WQ2Q4S, T-112K9S5H5, 111C2HNDW.',
    '- If you see a plausible shipment/load identifier in the OCR text, extract it exactly as written.',
    '- Do not set load_number to null if a plausible alphanumeric identifier is present.',
    '- Keep operational codes like IND2, MDW5, RDG1, ELP1 inside location fields when they belong there, but do not confuse them with the load number.',
]
FIELD_LINES = [
    '- pickup_location and dropoff_location should be the location strings.',
    '- pickup_time and dropoff_time should be the time/date strings.',
    '- total_rate should be the full money amount for the load.',
    '- rate_per_mile should be the $.../mi value.',
]
FORMAT_LINES = [
    '- Preserve formatting as closely as possible.',
    '- Return JSON only, with no markdown and no explanation.',
    '- Use null if truly missing.',
    '- Do not invent values.',
]

def remove_lines(prompt, lines):
    for line in lines:
        assert prompt.count(line) == 1, f'Expected exactly one instruction: {line}'
        prompt = prompt.replace(line+'\n','')
    return prompt

def make_prompt(variant, image_name, ocr_text):
    if variant == 'A4_P1_controlled':
        return p1(image_name, ocr_text)
    prompt = p2(image_name, ocr_text)
    if variant == 'A1_no_load_guidance': return remove_lines(prompt, LOAD_LINES)
    if variant == 'A2_no_field_guidance': return remove_lines(prompt, FIELD_LINES)
    if variant == 'A3_no_format_null_rules': return remove_lines(prompt, FORMAT_LINES)
    if variant == 'A0_full_P2': return prompt
    raise ValueError(variant)

example = ocr_rows[0]
base = make_prompt(VARIANTS[0], example['source_image'], example['ocr_text'])
for v in VARIANTS[1:]:
    current = make_prompt(v, example['source_image'], example['ocr_text'])
    print('\n###',v,'\n')
    print(''.join(difflib.unified_diff(base.splitlines(True),current.splitlines(True),fromfile='A0',tofile=v))[:4500])
assert all(make_prompt(v,example['source_image'],example['ocr_text']) for v in VARIANTS)

## Ground truth resolution
The expected dataset has `train.json`, `val.json`, and `test.json`, each a list of records containing `source_image` and the seven target fields. The notebook searches common repository locations. If the files live elsewhere, set `GT_DIR` explicitly. The evaluation uses `(split, source_image)` as a composite key.

In [ ]:
CANDIDATE_GT_DIRS = [
    ROOT / '../LoREx-160/rate_confirmations',
    ROOT / '../LoREx-160/rate_confirmations/labels',
    ROOT / 'data', ROOT / 'dataset', ROOT / 'datasets', ROOT,
]
GT_DIR = next((p for p in CANDIDATE_GT_DIRS if all((p / f'{s}.json').exists() for s in ['train','val','test'])), None)
if GT_DIR is None:
    print('Ground-truth files not auto-located. Set GT_DIR = Path("/absolute/path/to/split/json/files")')
else:
    print('Ground truth directory:', GT_DIR)

def load_ground_truth(gt_dir):
    if gt_dir is None: raise FileNotFoundError('Set GT_DIR to the directory containing train.json, val.json, test.json')
    gt = {}
    for split in ['train','val','test']:
        rows = json.loads((Path(gt_dir) / f'{split}.json').read_text(encoding='utf-8'))
        assert isinstance(rows,list), f'Expected list: {split}.json'
        for row in rows:
            key = (split,row['source_image'])
            assert key not in gt, f'Duplicate GT: {key}'
            gt[key] = row
    missing = set(keys) - set(gt)
    assert not missing, f'Missing GT for {len(missing)} matched records; examples: {list(missing)[:5]}'
    return gt

# Ground truth can be loaded after inference if files are not yet present.


## Inference with durable raw-response logging
Each `(variant, split, source_image)` has one JSON record saved immediately. Network failures remain `api_failed` and are retried on subsequent notebook runs. Valid responses are stored as `raw_output` even if parsing fails. Running with `LIMIT=135` reuses successful pilot records automatically. **No failed response is silently converted into seven null fields.**

In [ ]:
def recovery_parse(raw):
    text = raw.strip()
    text = re.sub(r'^```json\s*', '', text)
    text = re.sub(r'^```\s*', '', text)
    text = re.sub(r'\s*```$', '', text).strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match: raise ValueError('No JSON object')
        obj = json.loads(match.group(0))
    if not isinstance(obj,dict): raise ValueError('Expected JSON object')
    return obj

def strict_parse(raw):
    obj = json.loads(raw.strip())
    if not isinstance(obj,dict): raise ValueError('Expected JSON object')
    return obj

def atomic_write(path, obj):
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding='utf-8')
    temp.replace(path)

RESULT_FILE = OUT_DIR / 'raw_and_parsed.json'
existing = json.loads(RESULT_FILE.read_text(encoding='utf-8')) if RESULT_FILE.exists() else []
record_map = {(r['variant'],r['split'],r['source_image']):r for r in existing}
print('Previously recorded:',len(record_map))

def run_inference(limit=LIMIT):
    token = os.getenv('HF_TOKEN')
    if not token: raise ValueError('HF_TOKEN not found in .env')
    client = InferenceClient(api_key=token, provider=PROVIDER)
    for variant in VARIANTS:
        for index, row in enumerate(ocr_rows[:limit],1):
            k = (variant,row['split'],row['source_image'])
            if k in record_map and record_map[k].get('api_status') == 'ok': continue
            prompt = make_prompt(variant,row['source_image'],row['ocr_text'])
            result = {'variant':variant,'split':row['split'],'source_image':row['source_image'],
                      'model':MODEL,'provider':PROVIDER,'temperature':TEMPERATURE,'max_tokens':MAX_TOKENS,
                      'prompt':prompt,'raw_output':None,'api_status':'api_failed',
                      'parse_status':'not_attempted','parsed':None,'error':None,
                      'timestamp_utc':datetime.now(timezone.utc).isoformat()}
            for attempt in range(1,4):
                try:
                    completion = client.chat.completions.create(model=MODEL,
                        messages=[{'role':'user','content':prompt}],max_tokens=MAX_TOKENS,temperature=TEMPERATURE)
                    raw = completion.choices[0].message.content
                    if not isinstance(raw,str): raise ValueError('Empty/non-text model response')
                    result['raw_output'] = raw
                    result['api_status'] = 'ok'
                    result['error'] = None
                    break
                except Exception as exc:
                    result['error'] = f'{type(exc).__name__}: {exc}'
                    if attempt < 3: time.sleep(3*attempt)
            if result['api_status'] == 'ok':
                try:
                    obj = recovery_parse(result['raw_output'])
                    result['parsed'] = {f:obj.get(f) for f in FIELDS}
                    result['parse_status'] = 'ok'
                except Exception as exc:
                    result['parse_status'] = 'failed'
                    result['error'] = f'{type(exc).__name__}: {exc}'
            record_map[k] = result
            atomic_write(RESULT_FILE,list(record_map.values()))
            print(f'{variant} {index}/{limit} {row["split"]}/{row["source_image"]}: {result["api_status"]}/{result["parse_status"]}',flush=True)
    print('Saved:',RESULT_FILE)

# Uncomment to run paid/remote inference. Start with LIMIT=10.
# run_inference()


## Evaluation
Normalization below is intentionally explicit: lowercase, strip outer whitespace, and collapse internal whitespace; punctuation and currency symbols are preserved. **Field-level micro F1** treats a nonempty exact match as TP, an incorrect nonempty prediction as FP+FN, a missing prediction for a nonempty reference as FN, and a prediction where reference is empty as FP. Both-empty pairs are ignored for P/R/F1 but counted as correct for EM. Fuzzy similarity uses `difflib.SequenceMatcher`; normalized Levenshtein similarity uses edit distance divided by maximum string length. These definitions may differ from the manuscript's existing evaluation notebook: reconcile before reporting results as directly comparable.

In [ ]:
def norm(value):
    if value is None: return ''
    return ' '.join(str(value).lower().strip().split())

def edit_distance(a,b):
    if len(a)<len(b): a,b=b,a
    previous=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        current=[i]
        for j,cb in enumerate(b,1):
            current.append(min(current[-1]+1,previous[j]+1,previous[j-1]+(ca!=cb)))
        previous=current
    return previous[-1]

def score_pairs(pairs):
    tp=fp=fn=0
    exact=[]; fuzzy=[]; lev=[]
    for reference,prediction in pairs:
        a,b=norm(reference),norm(prediction)
        exact.append(float(a==b))
        fuzzy.append(difflib.SequenceMatcher(None,a,b).ratio()*100)
        denom=max(len(a),len(b))
        lev.append((1-edit_distance(a,b)/denom)*100 if denom else 100.0)
        if not a and not b: continue
        if a and b and a==b: tp+=1
        else:
            if b: fp+=1
            if a: fn+=1
    precision=tp/(tp+fp) if tp+fp else 0
    recall=tp/(tp+fn) if tp+fn else 0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0
    return {'N_pairs':len(pairs),'EM_pct':round(mean(exact)*100,2) if exact else None,
            'precision_pct':round(precision*100,2),'recall_pct':round(recall*100,2),
            'F1_pct':round(f1*100,2),'fuzzy_pct':round(mean(fuzzy),2) if fuzzy else None,
            'lev_pct':round(mean(lev),2) if lev else None,
            'TP':tp,'FP':fp,'FN':fn}

def evaluated_record(record,parser):
    if not record or record.get('api_status')!='ok': return None,'api_failed'
    if parser=='recovery':
        if record.get('parse_status')!='ok': return None,'parse_failed'
        return record['parsed'],'ok'
    try:
        parsed = strict_parse(record['raw_output'])
        return {f:parsed.get(f) for f in FIELDS},'ok'
    except Exception: return None,'parse_failed'

def evaluate(limit=LIMIT):
    gt=load_ground_truth(GT_DIR)
    configs=[(v,'recovery') for v in VARIANTS]+[('A0_full_P2','strict')]
    summary=[]; by_field=[]; coverage=[]
    for variant,parser in configs:
        label='A5_P2_strict_parser' if parser=='strict' else variant
        field_pairs={f:[] for f in FIELDS}
        statuses=Counter()
        for row in ocr_rows[:limit]:
            key=(variant,row['split'],row['source_image'])
            pred,status=evaluated_record(record_map.get(key),parser)
            statuses[status]+=1
            if status!='ok': continue  # explicitly exclude failures from extraction quality
            ref=gt[(row['split'],row['source_image'])]
            for f in FIELDS: field_pairs[f].append((ref.get(f),pred.get(f)))
        all_pairs=[pair for f in FIELDS for pair in field_pairs[f]]
        metrics=score_pairs(all_pairs)
        summary.append({'variant':label,'parser':parser,'n_requested':limit,
                        'n_success':statuses['ok'],'n_api_failed':statuses['api_failed'],
                        'n_parse_failed':statuses['parse_failed'],**metrics})
        for f in FIELDS:
            by_field.append({'variant':label,'field':f,**score_pairs(field_pairs[f])})
        coverage.append({'variant':label,**statuses})
    def save_csv(path,rows):
        with path.open('w',newline='',encoding='utf-8') as fp:
            w=csv.DictWriter(fp,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    save_csv(OUT_DIR / f'summary_{limit}.csv',summary)
    save_csv(OUT_DIR / f'fields_{limit}.csv',by_field)
    print('\nSummary (extraction scores only over successful parsed requests):')
    for r in summary:
        print(f"{r['variant']:27} success={r['n_success']:3}/{limit} API_fail={r['n_api_failed']:3} parse_fail={r['n_parse_failed']:3} EM={r['EM_pct']} F1={r['F1_pct']} fuzzy={r['fuzzy_pct']} lev={r['lev_pct']}")
    if len({r['n_success'] for r in summary})>1:
        print('WARNING: Unequal success counts. Compare on common successful document IDs before drawing prompt-quality conclusions.')
    print('CSV:',OUT_DIR / f'summary_{limit}.csv')
    return summary,by_field

# Uncomment after ground truth is located and inference has run:
# summary, by_field = evaluate()


## Common-success paired comparison (recommended for the paper)
If some variants fail, compare only documents where **all five inference variants** parsed successfully. This avoids a misleading result caused by unequal evaluated subsets. The A5 strict-parser failure rate should be reported separately as a parsing robustness outcome.

In [ ]:
def evaluate_common_success(limit=LIMIT):
    gt=load_ground_truth(GT_DIR)
    common=[]
    for row in ocr_rows[:limit]:
        key=(row['split'],row['source_image'])
        if all(evaluated_record(record_map.get((v,*key)),'recovery')[1]=='ok' for v in VARIANTS):
            common.append(row)
    print('Common successful documents:',len(common),'/',limit)
    if not common: return []
    output=[]
    for v in VARIANTS:
        pairs=[]
        for row in common:
            key=(row['split'],row['source_image'])
            pred,_=evaluated_record(record_map[(v,*key)],'recovery')
            for f in FIELDS: pairs.append((gt[key].get(f),pred.get(f)))
        output.append({'variant':v,'n_documents':len(common),**score_pairs(pairs)})
    path=OUT_DIR / f'paired_common_{limit}.csv'
    with path.open('w',newline='',encoding='utf-8') as fp:
        w=csv.DictWriter(fp,fieldnames=list(output[0]));w.writeheader();w.writerows(output)
    for r in output: print(r['variant'],'F1:',r['F1_pct'],'EM:',r['EM_pct'])
    print('Saved:',path)
    return output

# Uncomment after inference:
# paired = evaluate_common_success()


## Execution checklist
1. Verify prompt diffs and `GT_DIR`.
2. Set `LIMIT=10`, uncomment `run_inference()` and run; check the API/parse failures. Then run `evaluate()` and `evaluate_common_success()`.
3. For full coverage set `LIMIT=135` and rerun the inference/evaluation cells. The 10 previously completed records are reused.
4. Inspect field-level F1 and parsing failure counts. Do **not** describe any pilot numbers as final manuscript findings.
5. If the final paper needs *exactly* the earlier LoREx evaluation normalization, port its metric functions here before manuscript reporting. Note that A3 retains the JSON key schema, and A5 tests strict parsing on the *same* A0 raw responses.

**Reproducibility note:** Record the provider/model endpoint and date, because remote model serving can change. `temperature=0` reduces sampling variability but does not guarantee identical outputs across separate API runs.